In [18]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [19]:
def build_vedral_circuit(a, b):
    qc = QuantumCircuit(13, 5)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: qc.x(1)
    if a & 2: qc.x(4)
    if a & 4: qc.x(7)
    if a & 8: qc.x(10)

    # encode B
    if b & 1: qc.x(2)
    if b & 2: qc.x(5)
    if b & 4: qc.x(8)
    if b & 8: qc.x(11)

    qc.barrier()

    # adder gates
    qc.ccx(1,2,3)
    qc.ccx(4,5,6)
    qc.ccx(7,8,9)
    qc.ccx(10,11,12)
    
    qc.cx(1,2)
    qc.cx(4,5)
    qc.cx(7,8)
    qc.cx(10,11)
    
    qc.ccx(0,2,3)
    qc.ccx(3,5,6)
    qc.ccx(6,8,9)
    qc.ccx(9,11,12)
    
    qc.cx(10,11)
    qc.cx(10,11)
    qc.cx(9,11)
    
    qc.ccx(6,8,9)
    qc.cx(7,8)
    qc.ccx(7,8,9)
    qc.cx(7,8)
    qc.cx(6,8)
    
    qc.ccx(3,5,6)
    qc.cx(4,5)
    qc.ccx(4,5,6)
    qc.cx(4,5)
    qc.cx(3,5)
    
    qc.ccx(0,2,3)
    qc.cx(1,2)
    qc.ccx(1,2,3)
    qc.cx(1,2)
    qc.cx(0,2)
    
    # =====================================================
    # MEASUREMENTS
    # =====================================================
    
    
    qc.measure(2,0);
    qc.measure(5,1);
    qc.measure(8,2);
    qc.measure(11,3);
    qc.measure(12,4);

    return qc

In [20]:
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

# For research-grade accuracy, you usually want 10,000 shots, 
# but keeping it at 100 here for faster testing!
shots = 100

# ====================================================
# 1. HARDWARE INSTRUCTION SET (BASIS GATES)
# ====================================================
# We must define the physical gates the quantum hardware actually uses
basis_gates = ['cx', 'id', 'rz', 'sx', 'x']

# ====================================================
# 2. PHASE NOISE MODEL (Z-ERROR)
# ====================================================
p = 0.0021

# 1-Qubit Phase Error
error_1 = pauli_error([
    ('Z', p),
    ('I', 1-p)
])

# 2-Qubit Phase Error (Z ⊗ Z)
error_2 = error_1.tensor(error_1)

noise_model = NoiseModel()

# Apply phase noise to the physical single-qubit gates
# The 'sx' gate is the crucial one—it creates the superposition!
noise_model.add_all_qubit_quantum_error(error_1, ['sx', 'x', 'id'])

# Apply phase noise to the physical two-qubit gates
noise_model.add_all_qubit_quantum_error(error_2, ['cx'])

# Bind the noise model to the simulator
sim = AerSimulator(noise_model=noise_model)

# ====================================================
# 3. NUMPY ARRAYS FOR METRICS
# ====================================================
ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

# The maximum possible output for 4-bit + 4-bit is 30
D = 30

print("Running simulations, compiling circuits to basis gates...")

for a in range(16):
    for b in range(16):
        
        # Assume you have your build_vedral_circuit defined above
        qc = build_vedral_circuit(a, b)
        
        # CRITICAL STEP: Compile the high-level logic down to hardware basis gates
        compiled = transpile(qc, basis_gates=basis_gates)
        
        # Run the transpiled circuit
        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        # Ensure the binary string length matches your classical register size (e.g., 5 bits)
        correct_output = format(correct, '05b') 
        
        correct_counts = counts.get(correct_output, 0)
        
        # Error Rate calculation
        ER = 1 - correct_counts / shots
        
        total_ED = 0
        total_relative_ED = 0
        
        for output, freq in counts.items():
            noisy_decimal = int(output, 2)
            
            # Error Distance
            ED = abs(correct - noisy_decimal)
            total_ED += ED * freq
            
            if correct != 0:
                total_relative_ED += (ED / correct) * freq
                
        mean_ED = total_ED / shots
        
        NMED = mean_ED / D
        MRED = total_relative_ED / shots
        
        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# 4. OVERALL METRICS
# ====================================================
er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("\n--- Phase Noise Results ---")
print(f"ER (%) : {er * 100:.4f}")
print(f"NMED   : {nmed:.6f}")
print(f"MRED   : {mred:.6f}")

Running simulations, compiling circuits to basis gates...

--- Phase Noise Results ---
ER (%) : 7.4102
NMED   : 0.018328
MRED   : 0.050988


### Result is not so acuurate using FakeBackend by IBM

In [21]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

# Import the modern Qiskit Generic Backend
from qiskit.providers.fake_provider import GenericBackendV2

# ====================================================
# 1. INITIALIZE FAKE BACKEND & NOISE MODEL
# ====================================================
print("Generating a 13-qubit IBM hardware simulator...")

# This automatically builds a simulated backend with a realistic coupling map and noise
backend = GenericBackendV2(num_qubits=13)

# Extract the noise model, coupling map, and basis gates from the generated hardware
noise_model = NoiseModel.from_backend(backend)
coupling_map = backend.coupling_map
basis_gates = backend.operation_names

# Create the simulator using the hardware's exact noise profile
sim = AerSimulator(noise_model=noise_model)

def build_vedral_circuit(a, b):
    qc = QuantumCircuit(13, 5)

    # encode A
    if a & 1: qc.x(1)
    if a & 2: qc.x(4)
    if a & 4: qc.x(7)
    if a & 8: qc.x(10)

    # encode B
    if b & 1: qc.x(2)
    if b & 2: qc.x(5)
    if b & 4: qc.x(8)
    if b & 8: qc.x(11)

    qc.barrier()

    # adder gates
    qc.ccx(1, 2, 3)
    qc.ccx(4, 5, 6)
    qc.ccx(7, 8, 9)
    qc.ccx(10, 11, 12)
    
    qc.cx(1, 2)
    qc.cx(4, 5)
    qc.cx(7, 8)
    qc.cx(10, 11)
    
    qc.ccx(0, 2, 3)
    qc.ccx(3, 5, 6)
    qc.ccx(6, 8, 9)
    qc.ccx(9, 11, 12)
    
    qc.cx(10, 11)
    qc.cx(10, 11)
    qc.cx(9, 11)
    
    qc.ccx(6, 8, 9)
    qc.cx(7, 8)
    qc.ccx(7, 8, 9)
    qc.cx(7, 8)
    qc.cx(6, 8)
    
    qc.ccx(3, 5, 6)
    qc.cx(4, 5)
    qc.ccx(4, 5, 6)
    qc.cx(4, 5)
    qc.cx(3, 5)
    
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.ccx(1, 2, 3)
    qc.cx(1, 2)
    qc.cx(0, 2)
    
    # MEASUREMENTS
    qc.measure(2, 0)
    qc.measure(5, 1)
    qc.measure(8, 2)
    qc.measure(11, 3)
    qc.measure(12, 4)

    return qc

# ====================================================
# 2. NUMPY ARRAYS FOR METRICS
# ====================================================
shots = 10
ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

D = 30

print("Running simulations on hardware topography. This will take a moment...")

for a in range(16):
    for b in range(16):
        
        qc = build_vedral_circuit(a, b)
        
        # CRITICAL STEP: Transpile using the hardware's coupling map and basis gates
        # optimization_level=3 tries to minimize the number of SWAP gates required
        compiled = transpile(
            qc, 
            basis_gates=basis_gates, 
            coupling_map=coupling_map, 
            optimization_level=3
        )
        
        counts = sim.run(compiled, shots=shots).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b') 
        correct_counts = counts.get(correct_output, 0)
        
        ER = 1 - correct_counts / shots
        total_ED = 0
        total_relative_ED = 0
        
        for output, freq in counts.items():
            noisy_decimal = int(output, 2)
            ED = abs(correct - noisy_decimal)
            total_ED += ED * freq
            if correct != 0:
                total_relative_ED += (ED / correct) * freq
                
        mean_ED = total_ED / shots
        NMED = mean_ED / D
        MRED = total_relative_ED / shots
        
        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# 3. OVERALL METRICS
# ====================================================
er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("\n--- GenericBackendV2 Hardware Results ---")
print(f"ER (%) : {er * 100:.4f}")
print(f"NMED   : {nmed:.6f}")
print(f"MRED   : {mred:.6f}")

Generating a 13-qubit IBM hardware simulator...
Running simulations on hardware topography. This will take a moment...

--- GenericBackendV2 Hardware Results ---
ER (%) : 20.5469
NMED   : 0.036250
MRED   : 0.097733
